# GistKnit AI - Model Server (Stable version)
Ye notebook tumhara AI model chalata hai. Isme 3 sudhar hain:
1. **Model Google Drive me save hota hai**, to agli baar 10-15 minute ka download nahi, sirf 3-5 minute.
2. **Token Colab Secrets me rehta hai**, screenshot me kabhi nahi dikhega.
3. **Watchdog:** agar model ya ngrok beech me band ho jaye to apne aap dobara chalu hota hai.

**Pehli baar ka setup (sirf ek baar):**
- *Runtime > Change runtime type > T4 GPU* chuno, Save.
- Left side ka **chaabi (key) icon** dabao > *Add new secret* > Name: `NGROK_TOKEN`, Value: apna ngrok Authtoken, **Notebook access ON**.
- Phir *Runtime > Run all*. Drive ki permission maange to Allow karo.

**Roz ka kaam:** sirf *Run all*. Tab khula rakho.

In [ ]:
# 1) Install (1-2 minute)
!apt-get -qq install -y zstd pciutils lshw > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
!pip -q install pyngrok

In [ ]:
# 2) Drive + models + server start (pehli baar 15-20 minute, baad me 3-5 minute)
import os, subprocess, time, json, urllib.request
from google.colab import drive
drive.mount('/content/drive')

LOCAL = "/content/ollama_models"
BACKUP = "/content/drive/MyDrive/gistknit_models"
WITH_PHOTO_MODEL = True   # Drive me jagah kam ho (15 GB free) to False kar do
MODELS = ["qwen2.5:7b"] + (["qwen2.5vl:7b"] if WITH_PHOTO_MODEL else [])
os.makedirs(LOCAL, exist_ok=True)

if os.path.isdir(BACKUP) and os.listdir(BACKUP):
    print("Drive se models copy ho rahe hain (2-5 minute)...")
    subprocess.run(["cp", "-r", BACKUP + "/.", LOCAL + "/"])

subprocess.run(["pkill", "-f", "ollama serve"]); time.sleep(2)
env = dict(os.environ, OLLAMA_MODELS=LOCAL, OLLAMA_KEEP_ALIVE="24h", OLLAMA_HOST="0.0.0.0:11434",
           OLLAMA_CONTEXT_LENGTH="8192", OLLAMA_MAX_LOADED_MODELS="1")
subprocess.Popen(["ollama", "serve"], env=env, stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
time.sleep(8)

for m in MODELS:
    subprocess.run(["ollama", "pull", m])   # Drive se mil chuka ho to turant ho jata hai

os.makedirs(BACKUP, exist_ok=True)
r = subprocess.run(["cp", "-ru", LOCAL + "/.", BACKUP + "/"])
print("Models Drive me save ho gaye." if r.returncode == 0 else "Drive me save nahi hua (jagah kam ho sakti hai). Server phir bhi chalega.")

# model ko GPU me pehle se tayyar rakho, taaki pehla sawal tez aaye
req = urllib.request.Request("http://127.0.0.1:11434/api/generate",
    data=json.dumps({"model": "qwen2.5:7b", "prompt": "hi", "stream": False}).encode(),
    headers={"Content-Type": "application/json"})
urllib.request.urlopen(req, timeout=300).read()
print("Model GPU me tayyar hai.")

In [ ]:
# 3) Public address (ngrok). Token Colab Secrets se aata hai.
from pyngrok import ngrok
from google.colab import userdata

NGROK_DOMAIN = "handbrake-haiku-humorist.ngrok-free.dev"   # tumhara fixed address. Khali "" chhodo to random address milega.

try:
    token = userdata.get("NGROK_TOKEN")
except Exception:
    raise RuntimeError("Left side ke chaabi (key) icon me NGROK_TOKEN naam ka Secret banao, value me ngrok Authtoken daalo, 'Notebook access' ON karo. Phir ye cell dobara chalao.")

ngrok.kill()
ngrok.set_auth_token(token.strip())
tunnel = ngrok.connect(11434, domain=NGROK_DOMAIN) if NGROK_DOMAIN else ngrok.connect(11434)
url = tunnel.public_url
print("LLM_BASE_URL =", url + "/v1")
print("LLM_MODEL    = qwen2.5:7b")

In [ ]:
# 4) Test + GPU check ("100% GPU" dikhna chahiye)
req = urllib.request.Request(url + "/v1/chat/completions",
    data=json.dumps({"model": "qwen2.5:7b", "messages": [{"role": "user", "content": "Namaste, ek line me apna parichay do."}]}).encode(),
    headers={"Content-Type": "application/json", "ngrok-skip-browser-warning": "true"})
print(json.load(urllib.request.urlopen(req, timeout=180))["choices"][0]["message"]["content"])
!ollama ps

In [ ]:
# 5) Server chalta rakho + apne aap theek karo. Is cell ko chalta chhod do, tab khula rakho.
def alive(u):
    try:
        urllib.request.urlopen(u, timeout=8)
        return True
    except Exception:
        return False

while True:
    time.sleep(60)
    if not alive("http://127.0.0.1:11434/api/tags"):
        print("Ollama band mila, dobara chalu kar raha hu...")
        subprocess.Popen(["ollama", "serve"], env=env, stdout=open("/content/ollama.log", "a"), stderr=subprocess.STDOUT)
        time.sleep(10)
    if not ngrok.get_tunnels():
        print("ngrok band mila, dobara jod raha hu...")
        t = ngrok.connect(11434, domain=NGROK_DOMAIN) if NGROK_DOMAIN else ngrok.connect(11434)
        print("LLM_BASE_URL =", t.public_url + "/v1")
    print("OK", time.strftime("%H:%M"))